In [ ]:
import numpy as np
from scipy.special import jv, jn_zeros
from scipy.integrate import quad
from scipy.sparse.linalg import eigsh
from scipy.sparse import diags
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

# =============================================
# 1. Parámetros Físicos y de Discretización
# =============================================
D0 = 2e-9           # Coeficiente de difusión base [m²/s]
R = 1e-6            # Radio del cilindro [m]
z_c = 10e-9         # Altura crítica [m]
lambda_param = 0.2  # Parámetro de decaimiento [nm⁻¹]
kappa = 0.0         # Tasa de adsorción superficial [s⁻¹]
rho0 = 0.75e-6      # Posición radial inicial [m]
z0 = 15e-9          # Posición axial inicial [m]
n_modes_radial = 10 # Número de modos radiales
m_max = 10          # Número de modos axiales
N_z = 800           # Número de nodos en la dirección z
alpha_grid = 10.0   # Parámetro de estiramiento de la malla en z
delta = 1.0e-9      # Ancho de la transición en D(z)

# =============================================
# 2. Generación de la Malla Adaptativa en z
# =============================================
j = np.arange(N_z)
z = z_c * (1 + np.sinh(alpha_grid * (j / N_z - 0.5)) / np.sinh(0.5 * alpha_grid))

# =============================================
# 3. Función de Difusión D(z)
# =============================================
def D(z_val):
    S = 0.5 * (1 + np.tanh((z_val - z_c) / delta))
    return D0 * (np.exp(-lambda_param * z_val) * (1 - S) + S)

D_values = D(z)

# =============================================
# 4. Solución Radial (Funciones de Bessel y modos)
# =============================================
alpha_zeros = jn_zeros(0, n_modes_radial)
alpha_n = alpha_zeros / R

# Cálculo de la norma de cada modo radial (evitando extremos problemáticos)
norm_J0_sq = np.zeros(n_modes_radial)
for n in range(n_modes_radial):
    integrand = lambda r: r * jv(0, alpha_n[n] * r) ** 2
    norm_J0_sq[n] = quad(integrand, 0.01 * R, 0.99 * R, epsabs=1e-20, epsrel=1e-6)[0]

# =============================================
# 5. Solución Axial (Ensamblaje y autovalores)
# =============================================
eigenvalues_dict = {}
eigenvectors_dict = {}

h = np.diff(z)  # h[j] = z[j+1]-z[j] para j=0,...,N_z-2

for n in range(n_modes_radial):
    main_diag = np.empty(N_z)
    upper_diag = np.empty(N_z - 1)
    lower_diag = np.empty(N_z - 1)
    
    # Nodos internos (j=1,...,N_z-2)
    h_prev = h[:-1]       # z[j]-z[j-1] para j=1,...,N_z-2
    h_next = h[1:]        # z[j+1]-z[j] para j=1,...,N_z-2
    denom = (h_prev + h_next) / 2.0
    
    D_prev = 2 * D_values[1:-1] * D_values[:-2] / (D_values[1:-1] + D_values[:-2] + 1e-30)
    D_next = 2 * D_values[1:-1] * D_values[2:]   / (D_values[1:-1] + D_values[2:]   + 1e-30)
    
    coeff_prev = D_prev / (h_prev * denom)
    coeff_next = D_next / (h_next * denom)
    
    main_diag[1:-1] = coeff_prev + coeff_next + (alpha_n[n]**2) * D_values[1:-1]
    upper_diag[1:] = -coeff_next
    lower_diag[:-1] = -coeff_prev
    
    # Condición en z=0 (nodo 0): Robin
    h0 = h[0]
    main_diag[0] = D_values[0] / h0 + 1e-10 * D0
    upper_diag[0] = -D_values[0] / h0
    
    # Condición en z=L (nodo N_z-1): Dirichlet
    main_diag[-1] = 1.0
    main_diag += 1e-10 * D0  # Estabilización
    
    A = diags([main_diag, upper_diag, lower_diag], [0, 1, -1], format='csr')
    
    eigenvalues, eigenvectors = eigsh(A, k=m_max, which='SA', tol=1e-10,
                                      maxiter=200000, ncv=250)
    eigenvalues = np.clip(eigenvalues, 1e-20, 1e10)
    
    eigenvalues_dict[n] = eigenvalues
    eigenvectors_dict[n] = eigenvectors  # Cada columna es un modo axial para este modo radial

# =============================================
# 6. Cálculo de Coeficientes A_nm
# =============================================
def compute_A_nm(n):
    A_nm = np.zeros(m_max)
    Z_norms = np.zeros(m_max)
    for m in range(m_max):
        integrand = lambda z_val: (np.interp(z_val, z, eigenvectors_dict[n][:, m]) ** 2) / D(z_val)
        Z_norms[m], _ = quad(integrand, z[0], z[-1], epsabs=1e-20, epsrel=1e-8, limit=1000)
    Z_m_z0 = np.array([np.interp(z0, z, eigenvectors_dict[n][:, m]) for m in range(m_max)])
    J0_rho0 = jv(0, alpha_n[n] * rho0)
    valid_modes = (Z_norms > 1e-25) & (np.abs(Z_m_z0) > 1e-12)
    # Se incluye la normalización con la norma radial calculada
    A_nm = (J0_rho0 * Z_m_z0 * valid_modes) / (2 * np.pi * rho0 * norm_J0_sq[n] * (Z_norms + 1e-30))
    return A_nm

A_nm = Parallel(n_jobs=-1)(delayed(compute_A_nm)(n) for n in range(n_modes_radial))

# =============================================
# 7. Cálculo Optimizado de la Función de Correlación G(t)
# =============================================
def compute_G(t, N_quad=10):
    # Cuadratura en rho: nodos y pesos en [0.1R, 0.9R]
    rho_nodes, rho_weights = np.polynomial.legendre.leggauss(N_quad)
    rho_quad = 0.1 * R + 0.8 * R * (rho_nodes + 1) / 2.0
    weights_rho = rho_weights * (0.8 * R / 2.0)
    
    # Cuadratura en z: nodos y pesos en [z_c-5e-9, z_c+5e-9]
    z_nodes, z_weights = np.polynomial.legendre.leggauss(N_quad)
    z_quad = (z_c - 5e-9) + 10e-9 * (z_nodes + 1) / 2.0
    weights_z = z_weights * (10e-9 / 2.0)
    
    # Crear malla 2D para la cuadratura
    Rho, Z_grid = np.meshgrid(rho_quad, z_quad, indexing='ij')
    r2 = Rho ** 2 + Z_grid ** 2
    f_grid = (3 * Z_grid**2 - Rho**2) / (r2**2.5 + 1e-30)
    weights_2d = np.outer(weights_rho, weights_z)
    
    total = 0.0
    # Bucle sobre modos radiales
    for n in range(n_modes_radial):
        # Evaluar la función de Bessel en toda la malla
        J0_vals = jv(0, alpha_n[n] * Rho)  # Shape: (N_quad, N_quad)
        # Para cada modo axial m en el modo radial n, pre-evaluar los eigenvectores en z_quad
        # Se crea una matriz de evaluación: filas corresponden a z_quad, columnas a m (m_max)
        Z_evals = np.empty((N_quad, m_max))
        for m in range(m_max):
            # np.interp es vectorizado en z_quad
            Z_evals[:, m] = np.interp(z_quad, z, eigenvectors_dict[n][:, m])
        # Calcular el factor de decaimiento para cada modo axial en este modo n
        decay = np.exp(-(((alpha_n[n]**2) * D0) + eigenvalues_dict[n]) * t)  # Vector de tamaño (m_max,)
        # Sumar la contribución axial: para cada nodo de z_quad, suma sobre m
        # Realizamos la operación: sum_m[j] = sum_m A_nm[n][m] * Z_evals[j, m] * decay[m]
        axial_sum = np.dot(Z_evals, A_nm[n] * decay)  # Shape: (N_quad,)
        # La contribución total para este modo n es:
        contr = J0_vals * axial_sum  # Broadcasting axial_sum (1D) sobre la dimensión z en J0_vals
        total += np.sum(contr * f_grid * weights_2d)
        
    return (5 / (16 * np.pi)) * np.clip(total, 1e-40, 1e10)

# =============================================
# 8. Visualizaciones
# =============================================
def plot_all_results():
    # 8.a. Modos radiales normalizados
    plt.figure(figsize=(10, 6))
    rho_plot = np.linspace(0.01 * R, 0.99 * R, 200)
    for n in range(4):
        J0_vals = jv(0, alpha_n[n] * rho_plot)
        plt.plot(rho_plot / R, J0_vals / np.max(J0_vals), label=f'n={n + 1}')
    plt.xlabel(r'$\rho/R$', fontsize=12)
    plt.ylabel(r'$R_n(\rho)$', fontsize=12)
    plt.title('Modos Radiales Normalizados', fontsize=14)
    plt.legend()
    plt.grid(True, ls='--', alpha=0.7)
    plt.savefig('radial_modes_v4.pdf', bbox_inches='tight')
    plt.close()
    
    # 8.b. Modos axiales (ejemplo para algunos (n, m))
    fig, axs = plt.subplots(2, 2, figsize=(12, 8))
    for n in range(2):
        for m in range(2):
            mode = eigenvectors_dict[n][:, m]
            axs[n, m].plot(z * 1e9, mode / np.max(np.abs(mode)))
            axs[n, m].set_xlabel('z (nm)', fontsize=10)
            axs[n, m].set_title(f'Modo (n={n}, m={m})')
    plt.tight_layout()
    plt.savefig('axial_modes_v4.pdf', bbox_inches='tight')
    plt.close()
    
    # 8.c. Decaimiento temporal
    t_plot = np.logspace(-9, -3, 100)
    plt.figure(figsize=(10, 6))
    for n in range(2):
        for m in range(2):
            decay = np.exp(-(((alpha_n[n]**2) * D0) + eigenvalues_dict[n][m]) * t_plot)
            plt.loglog(t_plot, decay, label=f'n={n}, m={m}')
    plt.xlabel('Tiempo (s)', fontsize=12)
    plt.ylabel(r'$T_{nm}(t)$', fontsize=12)
    plt.title('Decaimiento Temporal', fontsize=14)
    plt.legend()
    plt.grid(True, which='both', ls='--')
    plt.savefig('temporal_decay_v4.pdf', bbox_inches='tight')
    plt.close()
    
    # 8.d. Visualización 3D de la densidad de probabilidad P(ρ, z, t)
    def plot_3d_density(t):
        rho_vals = np.linspace(0.1 * R, 0.9 * R, 40)
        z_vals = np.linspace(z_c - 5e-9, z_c + 5e-9, 40)
        Rg, Zg = np.meshgrid(rho_vals, z_vals, indexing='ij')
        P = np.zeros_like(Rg)
    
        for i in range(Rg.shape[0]):
            for j in range(Rg.shape[1]):
                sum_p = 0.0
                for n in range(n_modes_radial):
                    J0_val = jv(0, alpha_n[n] * Rg[i, j])
                    decay = np.exp(-(((alpha_n[n]**2) * D0) + eigenvalues_dict[n][0]) * t)
                    Z_val = np.interp(Zg[i, j], z, eigenvectors_dict[n][:, 0])
                    sum_p += A_nm[n][0] * J0_val * Z_val * decay
                P[i, j] = np.abs(sum_p)
    
        from mpl_toolkits.mplot3d import Axes3D
        fig = plt.figure(figsize=(14, 10))
        ax = fig.add_subplot(111, projection='3d')
        surf = ax.plot_surface(Rg * 1e6, Zg * 1e9, P, cmap='viridis',
                               rstride=1, cstride=1, alpha=0.8)
        ax.set_xlabel(r'$\rho$ (μm)', fontsize=12)
        ax.set_ylabel('z (nm)', fontsize=12)
        ax.set_zlabel(r'$P(\rho,z,t)$', fontsize=12)
        plt.title(f'Densidad de Probabilidad en t = {t:.1e} s', fontsize=14)
        fig.colorbar(surf, shrink=0.6, aspect=20)
        plt.savefig(f'density_t_{t:.0e}_v4.pdf', bbox_inches='tight')
        plt.close()
    
    for t_val in [1e-9, 1e-6, 1e-3]:
        plot_3d_density(t_val)

# =============================================
# 9. Ejecución Principal
# =============================================
if __name__ == "__main__":
    plot_all_results()
    
    t_values = np.logspace(-10, -1, 50)
    G_values = Parallel(n_jobs=-1)(delayed(compute_G)(t) for t in t_values)
    
    np.savetxt('G_vs_t_v4.dat', np.column_stack((t_values, G_values)),
               header='Tiempo(s) G(t)', fmt='%.3e')
    
    plt.figure(figsize=(10, 6))
    plt.loglog(t_values, G_values, 'bo-', label='Simulación')
    plt.plot(t_values, 1e-8 * t_values**-1.5, 'r--', label=r'Teoría $t^{-3/2}$')
    plt.xlabel('Tiempo (s)', fontsize=12)
    plt.ylabel('G(t)', fontsize=12)
    plt.title('Función de Correlación Nano-NMR', fontsize=14)
    plt.legend()
    plt.grid(True, which='both', ls='--')
    plt.savefig('Gt_result_v4.pdf', bbox_inches='tight')
    plt.close()